In [0]:
"""
Security Handler
================

Implements security measures:
1. Rate limiting (prevent abuse)
2. PII masking (protect customer data)
3. Input validation (prevent malicious input)
4. Audit logging (compliance tracking)
"""

import pandas as pd
from datetime import datetime, timedelta
from typing import Dict, Tuple, Optional
import re

# ============================================
# 1. SETUP AUDIT LOG TABLE
# ============================================

print("="*60)
print("SECURITY HANDLER: Protection Layer")
print("="*60)

# Create audit log table
print("\nStep 1: Setting up security infrastructure...")

spark.sql("USE banking_agent_db")

spark.sql("""
CREATE TABLE IF NOT EXISTS audit_log (
    log_id STRING,
    user_id STRING,
    action STRING,
    amount DOUBLE,
    status STRING,
    reason STRING,
    timestamp TIMESTAMP
)
USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS rate_limit_tracker (
    user_id STRING,
    request_count INT,
    window_start TIMESTAMP
)
USING DELTA
""")

print("✓ Audit log and rate limiting tables created")

# ============================================
# 2. SECURITY HANDLER CLASS
# ============================================

class SecurityHandler:
    """
    Implements security controls for the banking agent.
    
    Controls:
    1. Rate limiting (max 10 requests/minute per user)
    2. PII masking (hide sensitive data)
    3. Input validation (check amounts, formats)
    4. Audit logging (track all actions)
    """
    
    def __init__(self, rate_limit=10, time_window_minutes=1):
        """
        Initialize security handler.
        
        Args:
            rate_limit: Max requests allowed per time window
            time_window_minutes: Time window duration (minutes)
        """
        self.rate_limit = rate_limit
        self.time_window = timedelta(minutes=time_window_minutes)
    
    # ========================================
    # RATE LIMITING
    # ========================================
    
    def check_rate_limit(self, user_id: str) -> Tuple[bool, str]:
        """
        Check if user has exceeded rate limit.
        
        What it does:
        1. Get user's request count in current window
        2. If count < limit: Allow request
        3. If count >= limit: Block request
        
        Args:
            user_id: User identifier
            
        Returns:
            (is_allowed, message)
        """
        
        now = datetime.now()
        
        try:
            # Get user's current request count
            result = spark.sql(f"""
            SELECT request_count FROM rate_limit_tracker
            WHERE user_id = '{user_id}'
            AND window_start > '{now - self.time_window}'
            ORDER BY window_start DESC
            LIMIT 1
            """).collect()
            
            if result:
                current_count = result[0]['request_count']
            else:
                current_count = 0
            
            # Check if limit exceeded
            if current_count >= self.rate_limit:
                return False, f"Rate limit exceeded ({current_count}/{self.rate_limit})"
            
            # Increment counter
            new_count = current_count + 1
            
            # Update in table
            spark.sql(f"""
            INSERT INTO rate_limit_tracker VALUES ('{user_id}', {new_count}, '{now}')
            """)
            
            return True, f"Request allowed ({new_count}/{self.rate_limit})"
            
        except Exception as e:
            # Allow on error (fail open)
            return True, f"Rate check passed"
    
    # ========================================
    # PII MASKING
    # ========================================
    
    def mask_account_number(self, account_number: str) -> str:
        """
        Mask account number to protect privacy.
        
        What it does:
        - Show only last 4 digits
        - Hide first 6 digits
        
        Example:
            Input:  "1234567890"
            Output: "****567890"
            
        Why:
        - Enough info to identify customer (last 4)
        - Protects full account number in logs
        """
        
        if not account_number or len(account_number) < 4:
            return account_number
        
        # Keep last 4 digits, mask the rest
        masked = "*" * (len(account_number) - 4) + account_number[-4:]
        return masked
    
    def mask_pii(self, text: str) -> str:
        """
        Mask all PII in text.
        
        Masks:
        - Account numbers (10+ digits)
        - Phone numbers
        - Email addresses (partially)
        """
        
        # Mask account numbers (10 digit sequences)
        text = re.sub(r'\b\d{10,}\b', 
                     lambda m: self.mask_account_number(m.group(0)), 
                     text)
        
        # Mask phone numbers (10 digits)
        text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b',
                     '***-***-****',
                     text)
        
        # Mask email (partial - show first letter and domain)
        text = re.sub(r'\b([a-zA-Z])[a-zA-Z0-9._%+-]*@([a-zA-Z0-9.-]+)\b',
                     r'\1***@\2',
                     text)
        
        return text
    
    # ========================================
    # INPUT VALIDATION
    # ========================================
    
    def validate_amount(self, amount: float, max_amount: float = 25000) -> Tuple[bool, str]:
        """
        Validate transaction amount.
        
        Checks:
        1. Amount is positive
        2. Amount is not too large
        3. Amount is reasonable (not 0)
        
        Args:
            amount: Transaction amount
            max_amount: Maximum allowed amount
            
        Returns:
            (is_valid, message)
        """
        
        # Check if positive
        if amount <= 0:
            return False, f"Amount must be positive (got {amount})"
        
        # Check if exceeds limit
        if amount > max_amount:
            return False, f"Amount exceeds daily limit (${amount} > ${max_amount})"
        
        # Check if reasonable
        if amount > 1000:
            return True, f"⚠️ Large amount (${amount}) - will require verification"
        
        return True, f"Amount valid (${amount})"
    
    def validate_input(self, user_input: str) -> Tuple[bool, str]:
        """
        Validate user input for safety.
        
        Checks:
        1. No SQL injection attempts
        2. No script injection
        3. Reasonable length
        """
        
        # Check for SQL injection patterns
        dangerous_patterns = [
            r"';",           # SQL injection
            r"--",           # SQL comment
            r"/\*",          # SQL block comment
            r"DROP",         # Destructive SQL
            r"DELETE",       # Destructive SQL
            r"UPDATE",       # Potentially dangerous
            r"<script",      # Script injection
            r"javascript:",  # JavaScript injection
        ]
        
        for pattern in dangerous_patterns:
            if re.search(pattern, user_input, re.IGNORECASE):
                return False, f"Input contains potentially dangerous content"
        
        # Check length
        if len(user_input) > 1000:
            return False, f"Input too long (>{1000} chars)"
        
        return True, f"Input validated"
    
    # ========================================
    # AUDIT LOGGING
    # ========================================
    
    def log_action(self, user_id: str, action: str, 
                  amount: Optional[float] = None,
                  status: str = "success", 
                  reason: str = "") -> None:
        """
        Log action for compliance/audit.
        
        What gets logged:
        - Who: user_id
        - What: action (e.g., "transfer", "balance_check")
        - How much: amount (if applicable)
        - When: timestamp (automatic)
        - Result: status (success, failed, blocked)
        - Why: reason (if applicable)
        
        Why important:
        - Regulatory compliance (show what happened)
        - Fraud investigation (trace unauthorized activity)
        - Performance analysis (understand usage patterns)
        - Security audit (detect suspicious behavior)
        """
        
        import uuid
        from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
        
        log_id = f"log_{uuid.uuid4().hex[:12]}"
        now = datetime.now()
        
        try:
            # Ensure amount is always a float to match table schema
            amount_float = float(amount) if amount is not None else 0.0
            
            # Create DataFrame with explicit schema to avoid type mismatches
            schema = StructType([
                StructField("log_id", StringType()),
                StructField("user_id", StringType()),
                StructField("action", StringType()),
                StructField("amount", DoubleType()),
                StructField("status", StringType()),
                StructField("reason", StringType()),
                StructField("timestamp", TimestampType())
            ])
            
            log_data = spark.createDataFrame([
                (log_id, user_id, action, amount_float, status, reason, now)
            ], schema=schema)
            
            log_data.write.mode("append").saveAsTable("audit_log")
            
            # Print (masked for security)
            masked_reason = self.mask_pii(reason)
            print(f"  📝 Logged: {action} by {user_id} - {status} - {masked_reason}")
            
        except Exception as e:
            print(f"  ⚠️ Logging error: {e}")

# ============================================
# 3. TEST SECURITY HANDLER
# ============================================

print("\n" + "="*60)
print("TESTING SECURITY HANDLER")
print("="*60)

handler = SecurityHandler(rate_limit=3, time_window_minutes=1)

# Test 1: Rate Limiting
print("\nTest 1: Rate Limiting (max 3 requests/minute)")
print("-"*60)

user_id = "user_123"
for i in range(5):
    allowed, message = handler.check_rate_limit(user_id)
    status = "✓ ALLOWED" if allowed else "❌ BLOCKED"
    print(f"Request {i+1}: {status} - {message}")

# Test 2: PII Masking
print("\n" + "-"*60)
print("Test 2: PII Masking")
print("-"*60)

test_texts = [
    "Transfer from account 1234567890 to user",
    "Customer phone: 555-123-4567",
    "Email: john.doe@example.com"
]

for text in test_texts:
    masked = handler.mask_pii(text)
    print(f"Original: {text}")
    print(f"Masked:   {masked}\n")

# Test 3: Input Validation
print("-"*60)
print("Test 3: Input Validation")
print("-"*60)

test_amounts = [100, -50, 50000, 1500]

for amount in test_amounts:
    valid, message = handler.validate_amount(amount)
    status = "✓ VALID" if valid else "❌ INVALID"
    print(f"Amount ${amount}: {status} - {message}")

# Test 4: Input Sanitization
print("\n" + "-"*60)
print("Test 4: Input Sanitization")
print("-"*60)

test_inputs = [
    "Transfer $500 to savings",
    "Transfer $500'; DROP TABLE users; --",
    "<script>alert('xss')</script>"
]

for test_input in test_inputs:
    valid, message = handler.validate_input(test_input)
    status = "✓ SAFE" if valid else "❌ DANGEROUS"
    print(f"Input: {test_input[:50]}")
    print(f"Result: {status} - {message}\n")

# Test 5: Audit Logging
print("-"*60)
print("Test 5: Audit Logging")
print("-"*60)

print("Logging sample actions:")
handler.log_action("user_123", "transfer", amount=500, 
                  status="success", reason="Transfer to account 1234567890")
handler.log_action("user_456", "failed_login", 
                  status="blocked", reason="Exceeded rate limit")
handler.log_action("user_789", "balance_check", 
                  status="success", reason="Customer requested balance")

print("\n" + "="*60)
print("✓ SECURITY HANDLER READY")
print("="*60)

# Export
security_handler = handler

print(f"\n✓ Exported for Phase 3:")
print(f"  - security_handler (SecurityHandler instance)")
print(f"  - audit_log table (Databricks)")
print(f"  - rate_limit_tracker table (Databricks)")